# Day 26：LCEL 表达式语言与链式编排（`|`、Runnable、OutputParsers）

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第三阶段 Day 26！今天我们将从零构建一套兼容 LangChain LCEL 核心语义的可组合管道引擎（`v0.3.0-day26`），掌握 `Runnable`、`RunnableSequence`（`|` 运算符）、`RunnableParallel`、`RunnablePassthrough`、三大 `OutputParser` 以及 3 阶段「翻译 → 润色 → 摘要」流水线。

## 1. 环境初始化与导入 `project_iterations/day_26` 模块
自动定位仓库根目录并将 `project_iterations/day_26` 加入 `sys.path`。

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

day26_dir = repo_root / "project_iterations" / "day_26"
if str(day26_dir) not in sys.path:
    sys.path.insert(0, str(day26_dir))

from zhishu_agent import (
    AIMessage,
    ArticleSummary,
    JsonOutputParser,
    MockChatModel,
    PromptTemplate,
    PydanticOutputParser,
    Runnable,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
    StrOutputParser,
    build_translate_polish_summary_chain,
)

print("Day 26 模块导入成功！路径:", day26_dir)

Day 26 模块导入成功！路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_26


## 2. `Runnable`、`RunnableLambda` 与 `|` 管道运算符
演示如何用 `|` 将普通函数、自定义 `Runnable` 与 `Prompt | Model | Parser` 串联，并调用 `invoke`、`batch` 与 `stream`。

In [2]:
class AddPrefixRunnable(Runnable[str, str]):
    def __init__(self, prefix: str) -> None:
        super().__init__(name="AddPrefix")
        self.prefix = prefix

    def invoke(self, input_data: str, config=None) -> str:
        return f"{self.prefix}{input_data}"

text_chain = (lambda s: s.strip().upper()) | AddPrefixRunnable("[ZHISHU] ")
print("invoke:", text_chain.invoke("  lcel pipeline  "))
print("batch :", text_chain.batch(["  rag ", " agent "]))

prompt_chain = PromptTemplate("请概述核心特性：{topic}") | MockChatModel() | StrOutputParser()
print("Prompt | Model | StrOutputParser ->", prompt_chain.invoke({"topic": "LCEL 可组合管道"}))

invoke: [ZHISHU] LCEL PIPELINE
batch : ['[ZHISHU] RAG', '[ZHISHU] AGENT']
Prompt | Model | StrOutputParser -> 已收到指令：请概述核心特性：LCEL 可组合管道


## 3. `RunnablePassthrough.assign` 与 `RunnableParallel` 并行分支
演示字典在 `|` 中的自动强制转换，以及如何在保留原字段的基础上增量追加计算字段。

In [3]:
parallel = RunnableParallel(
    raw=RunnablePassthrough(),
    char_count=lambda d: len(d["text"]),
    words=lambda d: d["text"].split(),
)
print("RunnableParallel:", parallel.invoke({"text": "ZhiShu Agent LCEL"}))

enrich_chain = RunnablePassthrough.assign(
    upper=lambda d: d["text"].upper(),
    length=lambda d: len(d["text"]),
)
print("Passthrough.assign:", enrich_chain.invoke({"id": 101, "text": "hello lcel"}))

RunnableParallel: {'raw': {'text': 'ZhiShu Agent LCEL'}, 'char_count': 17, 'words': ['ZhiShu', 'Agent', 'LCEL']}
Passthrough.assign: {'id': 101, 'text': 'hello lcel', 'upper': 'HELLO LCEL', 'length': 10}


## 4. 三大 `OutputParser`：`StrOutputParser`、`JsonOutputParser`、`PydanticOutputParser`
演示自动剥离 ` ```json ``` ` 代码块并执行 Pydantic v2 强类型校验。

In [4]:
from pydantic import BaseModel, Field

class ServiceHealth(BaseModel):
    service: str = Field(description="服务名称")
    latency_ms: int = Field(ge=0, description="响应延迟毫秒")
    healthy: bool = Field(description="是否健康")

json_parser = JsonOutputParser()
pydantic_parser = PydanticOutputParser(ServiceHealth)

mock_output = AIMessage(content='```json\n{"service": "zhishu-rag", "latency_ms": 42, "healthy": true}\n```')
print("JsonOutputParser    :", json_parser.invoke(mock_output))
print("PydanticOutputParser:", repr(pydantic_parser.invoke(mock_output)))

JsonOutputParser    : {'service': 'zhishu-rag', 'latency_ms': 42, 'healthy': True}
PydanticOutputParser: ServiceHealth(service='zhishu-rag', latency_ms=42, healthy=True)


## 5. 实战演练：3 阶段「翻译 → 润色 → 结构化摘要」LCEL 链
将英文技术描述依次通过翻译、专业润色，最终解析为 `ArticleSummary` Pydantic 对象。

In [5]:
pipeline = build_translate_polish_summary_chain()
output = pipeline.invoke({
    "source_text": "ZhiShu Enterprise Agent Platform supports composable pipelines, automated operations, and secure data analytics."
})

print("阶段1 [翻译]:", output["translated_text"])
print("阶段2 [润色]:", output["polished_text"])
print("阶段3 [摘要]:", output["final_summary"].model_dump())
assert isinstance(output["final_summary"], ArticleSummary)

阶段1 [翻译]: 智枢企业级智能体平台支持可组合管道、自动化运维以及安全的数据分析。（译自：ZhiShu Enterprise Agent Platform supports composable pipelines, automated operations, and secure data analytics.）
阶段2 [润色]: 【专业润色版】智枢企业级智能体平台支持可组合管道、自动化运维以及安全的数据分析。 凭借标准化组件编排，助力企业实现高效、可靠的智能化升级。
阶段3 [摘要]: {'title': '智枢平台核心能力简报', 'summary': '【专业润色版】智枢企业级智能体平台支持可组合管道、自动化运维以及安全的数据分析。 凭借标准化组件...', 'keywords': ['智枢平台', '可组合管道', '企业级智能体'], 'word_count': 69}
